In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import os

load_dotenv()


engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    print("✅ OK connection")

✅ OK connection


In [2]:
from sqlalchemy import text

def drop_all_tables(engine):
    with engine.connect() as conn:
        conn.execute(text("""
            DROP TABLE IF EXISTS transactions CASCADE;
            DROP TABLE IF EXISTS comptes CASCADE;
            DROP TABLE IF EXISTS clients CASCADE;
            DROP TABLE IF EXISTS produits CASCADE;
            DROP TABLE IF EXISTS agences CASCADE;
            DROP TABLE IF EXISTS segments CASCADE;
            DROP TABLE IF EXISTS temps CASCADE;
        """))
        conn.commit()

# call
drop_all_tables(engine)

lire data

In [3]:

df = pd.read_csv("../data/financecore_clean.csv")


In [4]:
df['segment_client'].unique()

array(['Premium', 'Risque', 'Standard'], dtype=object)

séparation des tables avec insertion des données

In [5]:
from sqlalchemy import text

with open(r"C:\Users\user\Downloads\Bank\venv\sql\tables.sql", "r", encoding="utf-8") as file:
    sql_script = file.read()

with engine.begin() as conn:
    conn.execute(text(sql_script))

print("✅ tables created successfully")

✅ tables created successfully


In [6]:

# ======================
# FIX DATE (IMPORTANT FIX 🔥)
# ======================
df['date_transaction'] = pd.to_datetime(df['date_transaction']).dt.floor('D')

# ======================
# TEMPS TABLE
# ======================
temps_df = df[['date_transaction','annees','mois','trimestre','jour_semaine']].drop_duplicates()
temps_df = temps_df.reset_index(drop=True)
temps_df['id_temps'] = range(1, len(temps_df) + 1)

# 👉 mapping table (IMPORTANT FIX)
temps_map = temps_df[['date_transaction','id_temps']]
temps_df.to_sql('temps', engine, if_exists='append', index=False)
# ======================
# SEGMENTS
# ======================
segments_df = df[['segment_client']].drop_duplicates()
segments_df = segments_df.rename(columns={'segment_client':'nom_segment'})
segments_df['id_segment'] = range(1, len(segments_df)+1)

segments_df.to_sql('segments', engine, if_exists='append', index=False)

# ======================
# CLIENTS
# ======================
clients_df = df[['client_id','score_credit_client','segment_client','taux_rejet']].drop_duplicates()

clients_df = clients_df.merge(segments_df, left_on='segment_client', right_on='nom_segment')

clients_df['categorie_risque'] = clients_df['score_credit_client'].apply(
    lambda x: 'Low' if x>=700 else ('Medium' if x>=500 else 'High')
)

clients_df = clients_df.rename(columns={
    'client_id':'id_client',
    'score_credit_client':'score_credit'
})

clients_df = clients_df[['id_client','score_credit','categorie_risque','taux_rejet','id_segment']].drop_duplicates(subset='id_client')
clients_df['taux_rejet'] = clients_df['taux_rejet'] / 100

clients_df.to_sql('clients', engine, if_exists='append', index=False)

# ======================
# COMPTES
# ======================
comptes_df = df[['client_id','solde_avant']].drop_duplicates()
comptes_df['id_compte'] = "CPT_" + comptes_df['client_id'].astype(str)

comptes_df = comptes_df.rename(columns={
    'client_id':'id_client',
    'solde_avant':'solde'
})
comptes_df=comptes_df.drop_duplicates(subset='id_compte')
comptes_df.to_sql('comptes', engine, if_exists='append', index=False)

# ======================
# PRODUITS
# ======================
produits_df = df[['produit','categorie']].drop_duplicates()
produits_df['id_produit'] = range(1, len(produits_df)+1)
produits_df = produits_df.rename(columns={'produit':'nom_produit'})

produits_df.to_sql('produits', engine, if_exists='append', index=False)

# ======================
# AGENCES
# ======================
agences_df = df[['agence']].drop_duplicates()
agences_df['id_agence'] = range(1, len(agences_df)+1)
agences_df = agences_df.rename(columns={'agence':'nom_agence'})

agences_df.to_sql('agences', engine, if_exists='append', index=False)

# ======================
# TRANSACTIONS (FIXED 🔥)
# ======================

df['id_compte'] = "CPT_" + df['client_id'].astype(str)

# safe merges
df = df.merge(produits_df, left_on='produit', right_on='nom_produit', how='left')
df = df.merge(agences_df, left_on='agence', right_on='nom_agence', how='left')

# 👉 IMPORTANT FIX (NO LOSS MERGE)
df = df.merge(temps_map, on='date_transaction', how='left')

# ======================
# FINAL TRANSACTIONS TABLE
# ======================
transactions_df = df[[
    'transaction_id',
    'id_compte',
    'id_produit',
    'id_agence',
    'id_temps',
    'montant',
    'devise',
    'montant_eur',
    'type_operation',
    'statut'
]]

transactions_df = transactions_df.rename(columns={
    'transaction_id':'id_transactions'
})

# 🔥 DEBUG (IMPORTANT)
print("transactions shape:", transactions_df.shape)
print("missing temps:", transactions_df['id_temps'].isna().sum())

# ======================
# INSERT
# ======================
transactions_df=transactions_df.drop_duplicates(subset='id_transactions')
transactions_df.to_sql('transactions', engine, if_exists='append', index=False)

print("🎉 Pipeline ETL OK")

transactions shape: (15640, 10)
missing temps: 0
🎉 Pipeline ETL OK


les index

In [7]:
from sqlalchemy import text

with open(r"C:\Users\user\Downloads\Bank\venv\sql\indexes.sql", "r", encoding="utf-8") as file:
    sql_script = file.read()

with engine.begin() as conn:
    conn.execute(text(sql_script))

print("✅ Index created successfully")

✅ Index created successfully


les view

In [8]:
from sqlalchemy import text

with open(r"C:\Users\user\Downloads\Bank\venv\sql\view.sql", "r", encoding="utf-8") as file:
    sql_script = file.read()

with engine.begin() as conn:
    conn.execute(text(sql_script))

print("✅ view created successfully")

✅ view created successfully


conflit

In [9]:
from sqlalchemy import text

with open(r"C:\Users\user\Downloads\Bank\venv\sql\conflict.sql", "r", encoding="utf-8") as file:
    sql_script = file.read()

with engine.begin() as conn:
    conn.execute(text(sql_script))

print("✅ conflict created successfully")

✅ conflict created successfully


vérification

In [10]:
from sqlalchemy import text

with open(r"C:\Users\user\Downloads\Bank\venv\sql\vérification.sql", "r", encoding="utf-8") as file:
    sql_script = file.read()

with engine.begin() as conn:
    conn.execute(text(sql_script))

print("✅ view created successfully")

✅ view created successfully
